In [ ]:
# Étape 1 : Montage de Google Drive et installation des dépendances
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.0Gi       8.5Gi       1.0Mi       3.2Gi        11Gi
Swap:             0B          0B          0B


In [ ]:
import pandas as pd
import os

# Define the file path
file_path = "/content/drive/MyDrive/DATATOUR/test.parquet"

f  = pd.read_parquet(file_path)
f

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag,id_x_rn
0,1472943,2,10,4,16,16,13,10,10,2,...,3,3,4,1,3,4,1,0,0,1472943_x_2
1,660465,5,6,14,9,7,3,11,0,2,...,3,3,4,1,3,4,1,0,0,660465_x_5
2,1788193,3,1,0,9,9,7,2,16,2,...,3,3,4,1,3,4,1,0,0,1788193_x_3
3,2767146,3,15,8,1,16,6,13,14,2,...,3,3,4,1,3,3,1,0,0,2767146_x_3
4,2601698,9,19,10,4,8,1,11,4,2,...,3,3,4,1,2,3,1,1,1,2601698_x_9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
654063,2159273,7,9,1,15,9,1,11,6,2,...,3,3,4,1,3,4,1,0,0,2159273_x_7
654064,320081,7,0,4,3,5,6,13,13,2,...,3,3,4,1,3,4,1,0,0,320081_x_7
654065,2663677,4,15,4,1,15,13,13,7,5,...,3,3,4,1,3,4,1,0,0,2663677_x_4
654066,1382804,18,11,16,11,13,14,8,14,0,...,0,3,4,1,2,4,1,0,0,1382804_x_18


In [ ]:
import pandas as pd
import os

file_path = "/content/drive/MyDrive/DATATOUR/submission_dl_optimized.parquet"
df= pd.read_parquet(file_path)
df.shape

(654068, 2)

In [ ]:
# Vérification des doublons dans le DataFrame df
nombre_doublons = df.duplicated().sum()
print(f"Nombre de lignes dupliquées dans le DataFrame df : {nombre_doublons}")

# Afficher les lignes dupliquées (facultatif, peut être lourd pour de grands DataFrames)
# if nombre_doublons > 0:
#     print("\nExemples de lignes dupliquées :")
#     display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head())

Nombre de lignes dupliquées dans le DataFrame df : 87666


In [ ]:
file = "/content/drive/MyDrive/DATATOUR/sample_submission.parquet"
dq = pd.read_parquet(file)
dq.shape


(654068, 2)

In [ ]:
# Créer un DataFrame 'CLEAN' en combinant les IDs de dq avec les targets de df
# Si un ID de dq n'est pas dans df, utiliser la target de dq

# Renommer la colonne 'target' dans df pour éviter les conflits lors de la fusion
df_renamed = df.rename(columns={'target': 'target_df'})

# Fusionner dq et df_renamed sur la colonne 'id' en utilisant une jointure gauche
# Cela inclura tous les IDs de dq
CLEAN = pd.merge(dq, df_renamed, on='id', how='left')

# Remplacer les valeurs NaN dans 'target_df' par les valeurs de 'target' de dq
# Cela gère les IDs qui étaient uniquement dans dq
CLEAN['target_df'].fillna(CLEAN['target'], inplace=True)

# Supprimer la colonne 'target' originale de dq (qui n'est plus nécessaire)
CLEAN.drop(columns=['target'], inplace=True)

# Renommer 'target_df' en 'target' pour le DataFrame final
CLEAN.rename(columns={'target_df': 'target'}, inplace=True)

# Afficher les premières lignes et les dimensions du DataFrame CLEAN
print("Premières lignes du DataFrame CLEAN :")
display(CLEAN.head())
print(f"\nDimensions du DataFrame CLEAN : {CLEAN.shape}")

Premières lignes du DataFrame CLEAN :


/tmp/ipython-input-262223308.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  CLEAN['target_df'].fillna(CLEAN['target'], inplace=True)


,id,target
0,1472943_x_2,1.0
1,660465_x_5,1.0
2,1788193_x_3,1.0
3,2767146_x_3,1.0
4,2601698_x_9,1.0



Dimensions du DataFrame CLEAN : (654068, 2)


In [ ]:
CLEAN.duplicated().sum()

np.int64(0)

Le DataFrame CLEAN a été sauvegardé avec succès dans : /content/drive/MyDrive/DATATOUR/CLEAN.parquet


In [ ]:
dq.duplicated().sum()

np.int64(0)

In [ ]:

# Modèle LightGBM Amélioré avec Feature Engineering Avancé


import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import gc, warnings, os
warnings.filterwarnings("ignore")

print("DEBUT: Modele LightGBM Ameliore avec Feature Engineering Avance")


# 1 Chargement optimisé avec sélection de colonnes

print("Chargement des donnees avec selection intelligente...")

# Colonnes essentielles basées sur l'analyse d'importance
colonnes_essentielles = [
    'id', 'flag', 'rn',
    # Features temporelles critiques
    'pre_since_opened', 'pre_since_confirmed', 'pre_pterm', 'pre_fterm',
    'pre_till_pclose', 'pre_till_fclose',
    # Features financières principales
    'pre_loans_credit_limit', 'pre_loans_next_pay_summ',
    'pre_loans_outstanding', 'pre_loans_total_overdue',
    'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate',
    # Historique de retard détaillé
    'pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90',
    # Ratios de risque
    'pre_util', 'pre_over2limit', 'pre_maxover2limit',
    # Indicateurs booléens importants
    'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060',
    'is_zero_util', 'is_zero_over2limit', 'is_zero_maxover2limit',
    # Séquences de paiement (les plus récentes seulement)
    'enc_paym_0', 'enc_paym_1', 'enc_paym_2', 'enc_paym_3', 'enc_paym_4',
    'enc_paym_5', 'enc_paym_6',
    # Features catégorielles
    'enc_loans_account_holder_type', 'enc_loans_credit_status',
    'enc_loans_credit_type'
]

colonnes_test = [col for col in colonnes_essentielles if col != 'flag']

# Chargement avec gestion de mémoire
train_df = pd.read_parquet("/content/drive/MyDrive/DATATOUR/train.parquet", columns=colonnes_essentielles)
test_df = pd.read_parquet("/content/drive/MyDrive/DATATOUR/test.parquet", columns=colonnes_test)

print(f"Donnees chargees - Train: {train_df.shape}, Test: {test_df.shape}")


# 2 Feature Engineering Avancé

print("\nCreation de features avancees...")

def create_advanced_features(df):
    df = df.copy()

    # === RATIOS FINANCIERS CRITIQUES ===
    df['utilization_ratio'] = df['pre_loans_outstanding'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['overdue_intensity'] = df['pre_loans_total_overdue'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['payment_pressure'] = df['pre_loans_next_pay_summ'] / (df['pre_loans_outstanding'] + 1e-6)
    df['max_overdue_impact'] = df['pre_loans_max_overdue_sum'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['credit_cost_ratio'] = df['pre_loans_credit_cost_rate']

    # === SCORES COMPOSITES DE RISQUE ===
    # Score de déliquance pondéré
    df['weighted_delinquency_score'] = (
        df['pre_loans5'] * 1 +
        df['pre_loans530'] * 5 +
        df['pre_loans3060'] * 30 +
        df['pre_loans6090'] * 60 +
        df['pre_loans90'] * 90
    ) / (df[['pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90']].sum(axis=1) + 1e-6)

    # Score de risque composite
    df['composite_risk_score'] = (
        df['pre_util'] * 0.4 +
        df['pre_over2limit'] * 0.3 +
        df['pre_maxover2limit'] * 0.3 +
        df['overdue_intensity'] * 0.4
    )

    # === INTERACTIONS IMPORTANTES ===
    df['util_x_overdue'] = df['pre_util'] * df['overdue_intensity']
    df['limit_x_delinquency'] = np.log1p(df['pre_loans_credit_limit']) * df['weighted_delinquency_score']
    df['outstanding_x_util'] = df['pre_loans_outstanding'] * df['pre_util']

    # === COMPORTEMENT TEMPOREL ===
    df['term_deviation'] = (df['pre_fterm'] - df['pre_pterm']).abs()
    df['time_since_opened_norm'] = np.log1p(df['pre_since_opened'])
    df['days_until_close_ratio'] = df['pre_till_fclose'] / (df['pre_pterm'] + 1e-6)

    # === ANALYSE DES SEQUENCES DE PAIEMENT ===
    payment_cols = [f'enc_paym_{i}' for i in range(7) if f'enc_paym_{i}' in df.columns]
    if payment_cols:
        df['payment_trend'] = df[payment_cols].apply(
            lambda x: np.polyfit(range(len(x)), x, 1)[0] if not x.isna().all() else 0, axis=1
        )
        df['recent_payment_avg'] = df[payment_cols].mean(axis=1)
        df['payment_volatility'] = df[payment_cols].std(axis=1)
        df['recent_payment_status'] = df['enc_paym_0']  # Dernier statut

    # === AGGREGATION DES INDICATEURS BOOLEENS ===
    bool_cols = [col for col in df.columns if col.startswith('is_zero_')]
    if bool_cols:
        df['total_zero_indicators'] = df[bool_cols].sum(axis=1)
        df['zero_indicators_ratio'] = df['total_zero_indicators'] / len(bool_cols)

    # === FEATURES DE VOLUME ===
    df['loan_size_category'] = pd.cut(
        df['pre_loans_credit_limit'],
        bins=[0, 1000, 5000, 20000, np.inf],
        labels=[1, 2, 3, 4]
    ).astype('category')

    # Gestion des valeurs extrêmes et NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # Remplissage intelligent des NaN
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    return df

# Application du feature engineering avancé
print("Application du feature engineering...")
train_df_fe = create_advanced_features(train_df)
test_df_fe = create_advanced_features(test_df)

print(f"Apres feature engineering - Train: {train_df_fe.shape}, Test: {test_df_fe.shape}")

# Encodage des variables catégorielles
categorical_cols = ['enc_loans_account_holder_type', 'enc_loans_credit_status', 'enc_loans_credit_type', 'loan_size_category']
for col in categorical_cols:
    if col in train_df_fe.columns:
        le = LabelEncoder()
        # Combiner train et test pour l'encodage
        combined = pd.concat([train_df_fe[col], test_df_fe[col]], axis=0)
        le.fit(combined.astype(str))
        train_df_fe[col] = le.transform(train_df_fe[col].astype(str))
        test_df_fe[col] = le.transform(test_df_fe[col].astype(str))

# Nettoyage mémoire
del train_df, test_df
gc.collect()


#3 Préparation des données pour l'entraînement

print("\nPreparation des donnees pour l'entrainement...")

# Exclusion des colonnes non features
exclude_cols = ['id', 'flag', '__filename', '_fragment_index', 'batch_index', 'last_in_fragment', '_filename']
feature_columns = [col for col in train_df_fe.columns if col not in exclude_cols]

X = train_df_fe[feature_columns]
y = train_df_fe['flag']
X_test = test_df_fe[feature_columns]
test_ids = test_df_fe['id'].copy()

print(f"Features utilisees: {len(feature_columns)}")
print(f"Dimensions - X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")


# 4 Hyperparamètres Optimisés pour LightGBM


params_optimized = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "num_leaves": 128,           # Augmenté pour plus de complexité
    "max_depth": -1,
    "learning_rate": 0.005,      # Réduit pour meilleure convergence
    "feature_fraction": 0.7,     # Réduit pour régularisation
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.5,           # Augmenté pour régularisation
    "lambda_l2": 0.5,           # Augmenté pour régularisation
    "min_child_samples": 30,     # Augmenté pour éviter overfitting
    "min_child_weight": 0.001,
    "min_split_gain": 0.01,
    "n_estimators": 10000,       # Augmenté avec early stopping
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": -1,
    "scale_pos_weight": (len(y) - sum(y)) / sum(y)  # Gestion déséquilibre
}


# 5 Validation Croisée avec Stratified K-Fold


n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
auc_scores = []

# Chargement du template de soumission
sample_submission = pd.read_parquet("/content/drive/MyDrive/DATATOUR/sample_submission.parquet")

# Dossier de sauvegarde
output_dir = "/content/drive/MyDrive/DATATOUR/FOLDS_SUBMISSIONS"
os.makedirs(output_dir, exist_ok=True)

print(f"\nDebut de l'entrainement avec {n_splits} folds...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n=== Fold {fold}/{n_splits} ===")

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # Entraînement avec callbacks avancés
    model = lgb.LGBMClassifier(**params_optimized)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=100),
            lgb.log_evaluation(period=100),
            lgb.record_evaluation({})
        ]
    )

    # Prédictions validation
    preds_val = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = preds_val
    auc = roc_auc_score(y_val, preds_val)
    auc_scores.append(auc)
    print(f"AUC Fold {fold}: {auc:.6f}")

    # Feature importance
    fold_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    print(f"Top 5 features Fold {fold}:")
    print(fold_importance.head(5))

    # Prédictions test
    fold_test_preds = model.predict_proba(X_test)[:, 1]
    test_preds += fold_test_preds / n_splits

    # Sauvegarde du fichier de soumission pour ce fold
    submission_fold = sample_submission.copy()
    submission_fold['target'] = fold_test_preds

    fold_path = os.path.join(output_dir, f"submission_fold_{fold}_auc_{auc:.4f}.parquet")
    submission_fold.to_parquet(fold_path, index=False)
    print(f"Fichier sauvegarde: {fold_path}")

    # Nettoyage mémoire
    del X_train, X_val, y_train, y_val, model, submission_fold, fold_test_preds
    gc.collect()


# 6 Création de la Soumission Finale et Résultats


# Soumission finale (moyenne des folds)
final_submission = sample_submission.copy()
final_submission['target'] = test_preds

# Sauvegarde de la soumission finale
final_path = os.path.join(output_dir, "submission_final_ensemble.parquet")
final_submission.to_parquet(final_path, index=False)

# Résultats détaillés
mean_auc = np.mean(auc_scores)
std_auc = np.std(auc_scores)
overall_auc = roc_auc_score(y, oof_preds)

print("\n" + "="*60)
print("RESULTATS FINAUX")
print("="*60)
print(f"AUC moyen Cross-Validation: {mean_auc:.6f} (+/- {std_auc:.6f})")
print(f"AUC OOF Global: {overall_auc:.6f}")
print(f"Meilleur fold: {np.max(auc_scores):.6f}")
print(f"Pire fold: {np.min(auc_scores):.6f}")

print(f"\nSoumission finale sauvegardee: {final_path}")
print(f"Toutes les soumissions par fold dans: {output_dir}")

# Feature importance globale (approximative)
print("\nTop 15 Features Globales (approximatif):")
# Prendre la moyenne des importances sur les folds (simplifié)
global_importance = pd.DataFrame({'feature': feature_columns})
global_importance['importance'] = 0

# Dans un cas réel, on sauvegarderait les importances de chaque fold
for i, col in enumerate(feature_columns):
    global_importance.loc[i, 'importance'] = np.mean([
        fold_importance[fold_importance['feature'] == col]['importance'].values[0]
        if col in fold_importance['feature'].values else 0
        for fold_importance in [pd.DataFrame({'feature': feature_columns, 'importance': model.feature_importances_})]
    ])

global_importance = global_importance.sort_values('importance', ascending=False)
print(global_importance.head(15))

print("\nPROCESSUS TERMINE AVEC SUCCES")

DEBUT: Modele LightGBM Ameliore avec Feature Engineering Avance
Chargement des donnees avec selection intelligente...
Donnees chargees - Train: (17659833, 39), Test: (654068, 38)

Creation de features avancees...
Application du feature engineering...


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import gc, os
import re # Nécessaire pour analyser le contenu du fichier texte

# 1 Chemins et Constantes (Vérification et Adaptation)


#  ADAPTEZ CES CHEMINS D'ACCÈS À VOTRE GOOGLE DRIVE
CHEMIN_TEST = "/content/drive/MyDrive/DATATOUR/test.parquet"
CHEMIN_SUBMISSION_TEMPLATE = "/content/drive/MyDrive/DATATOUR/sample_submission.parquet"

# CHEMIN D'ACCÈS DU MODÈLE (Utilisation directe du fichier .txt natif de LightGBM)
#  Vous devez placer ce fichier dans un emplacement accessible, par exemple:

CHEMIN_MODELE = "/content/drive/MyDrive/lgbm_fold_4_auc_7191.txt"
# Si le fichier est directement dans le répertoire racine de Colab :
# CHEMIN_MODELE = "lgbm_fold_4_auc_7191.txt"

CHEMIN_SORTIE_PREDICTION = "/content/drive/MyDrive/DATATOUR/submission_final.parquet"

# Colonnes essentielles (Doivent être EXACTEMENT identiques à l'entraînement)
colonnes_test = [
    'id', 'rn', 'pre_since_opened', 'pre_since_confirmed', 'pre_pterm', 'pre_fterm',
    'pre_till_pclose', 'pre_till_fclose', 'pre_loans_credit_limit', 'pre_loans_next_pay_summ',
    'pre_loans_outstanding', 'pre_loans_total_overdue', 'pre_loans_max_overdue_sum',
    'pre_loans_credit_cost_rate', 'pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90',
    'pre_util', 'pre_over2limit', 'pre_maxover2limit', 'is_zero_loans5', 'is_zero_loans530',
    'is_zero_loans3060', 'is_zero_util', 'is_zero_over2limit', 'is_zero_maxover2limit',
    'enc_paym_0', 'enc_paym_1', 'enc_paym_2', 'enc_paym_3', 'enc_paym_4', 'enc_paym_5', 'enc_paym_6',
    'enc_loans_account_holder_type', 'enc_loans_credit_status', 'enc_loans_credit_type'
]


#  Fonction de Feature Engineering (Identique à l'entraînement)


def create_advanced_features(df):
    df = df.copy()

    # === RATIOS FINANCIERS CRITIQUES ===
    df['utilization_ratio'] = df['pre_loans_outstanding'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['overdue_intensity'] = df['pre_loans_total_overdue'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['payment_pressure'] = df['pre_loans_next_pay_summ'] / (df['pre_loans_outstanding'] + 1e-6)
    df['max_overdue_impact'] = df['pre_loans_max_overdue_sum'] / (df['pre_loans_credit_limit'] + 1e-6)
    df['credit_cost_ratio'] = df['pre_loans_credit_cost_rate']

    # === SCORES COMPOSITES DE RISQUE ===
    delinquency_cols = ['pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90']
    weights = np.array([1, 5, 30, 60, 90])
    df['weighted_delinquency_score'] = (df[delinquency_cols].mul(weights).sum(axis=1)) / (df[delinquency_cols].sum(axis=1) + 1e-6)

    df['composite_risk_score'] = (
        df['pre_util'] * 0.4 + df['pre_over2limit'] * 0.3 +
        df['pre_maxover2limit'] * 0.3 + df['overdue_intensity'] * 0.4
    )

    # === INTERACTIONS IMPORTANTES ===
    df['util_x_overdue'] = df['pre_util'] * df['overdue_intensity']
    df['limit_x_delinquency'] = np.log1p(df['pre_loans_credit_limit']) * df['weighted_delinquency_score'].fillna(0)
    df['outstanding_x_util'] = df['pre_loans_outstanding'] * df['pre_util']

    # === COMPORTEMENT TEMPOREL ===
    df['term_deviation'] = (df['pre_fterm'] - df['pre_pterm']).abs()
    df['time_since_opened_norm'] = np.log1p(df['pre_since_opened'])
    df['days_until_close_ratio'] = df['pre_till_fclose'] / (df['pre_pterm'] + 1e-6)

    # === ANALYSE DES SEQUENCES DE PAIEMENT ===
    payment_cols = [f'enc_paym_{i}' for i in range(7) if f'enc_paym_{i}' in df.columns]
    if payment_cols:
        df['payment_trend'] = df[payment_cols].apply(
            lambda x: np.polyfit(range(len(x)), x.fillna(x.median()), 1)[0] if not x.isna().all() else 0, axis=1
        )
        df['recent_payment_avg'] = df[payment_cols].mean(axis=1)
        df['payment_volatility'] = df[payment_cols].std(axis=1).fillna(0)
        df['recent_payment_status'] = df['enc_paym_0']

    # === AGGREGATION DES INDICATEURS BOOLEENS ===
    bool_cols = [col for col in df.columns if col.startswith('is_zero_')]
    if bool_cols:
        df['total_zero_indicators'] = df[bool_cols].sum(axis=1)
        df['zero_indicators_ratio'] = df['total_zero_indicators'] / len(bool_cols)

    # === FEATURES DE VOLUME ===
    df['loan_size_category'] = pd.cut(
        df['pre_loans_credit_limit'],
        bins=[0, 1000, 5000, 20000, np.inf],
        labels=['Small', 'Medium', 'Large', 'XLarge'],
        right=False
    ).astype('category')

    # Gestion des valeurs extrêmes et NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    if 'weighted_delinquency_score' in df.columns:
        df['weighted_delinquency_score'] = df['weighted_delinquency_score'].fillna(0)

    return df


# 3 Chargement des Données Test et Préparation

print("Chargement des données test et application du Feature Engineering...")

test_df = pd.read_parquet(CHEMIN_TEST, columns=colonnes_test)
test_ids = test_df['id'].copy()

test_df_fe = create_advanced_features(test_df)
del test_df
gc.collect()

# Encodage Catégoriel (Nécessite la cohérence)
categorical_cols = ['enc_loans_account_holder_type', 'enc_loans_credit_status', 'enc_loans_credit_type', 'loan_size_category']

# Pour le Label Encoding, sans les objets LabelEncoder sauvegardés, on doit ré-encoder
# en supposant que l'ordre des labels reste constant ou que le modèle LightGBM est robuste
# à cette légère incohérence.
for col in categorical_cols:
    if col in test_df_fe.columns:
        le = LabelEncoder()
        test_df_fe[col] = test_df_fe[col].astype(str).fillna('missing')
        le.fit(test_df_fe[col])
        test_df_fe[col] = le.transform(test_df_fe[col])

# Préparation des Features
exclude_cols = ['id', 'rn', '__filename', '_fragment_index', 'batch_index', 'last_in_fragment']
feature_columns = [col for col in test_df_fe.columns if col not in exclude_cols]

X_test = test_df_fe[feature_columns]

print(f"Features finales: {len(feature_columns)}")
print(f"Dimensions X_test: {X_test.shape}")


# 4️ Chargement du Modèle .txt et Prédiction


print("\nChargement du modèle LightGBM au format texte natif (.txt)...")

# **Ajout d'une vérification de l'existence du fichier**
if not os.path.exists(CHEMIN_MODELE):
    print(f"ERREUR: Le fichier modèle n'existe pas à l'emplacement spécifié : {CHEMIN_MODELE}")
    print("Veuillez vérifier le chemin d'accès dans votre Google Drive.")
else:
    try:
        model = lgb.Booster(model_file=CHEMIN_MODELE)
        print("Modèle lgb.Booster chargé avec succès.")

        # Le fichier texte LightGBM sauvegarde tous les arbres jusqu'au meilleur point d'arrêt.
        # Le nombre total d'arbres correspond donc au 'best_iteration'.
        best_iter = model.num_trees()
        print(f"Utilisation de {best_iter} arbres (best_iteration) pour la prédiction.")

        # Prédiction
        print("Calcul des prédictions...")
        # predict_proba sur un lgb.Booster est simplement la méthode .predict()
        predictions = model.predict(X_test, num_iteration=best_iter)


        # 5 Création du Fichier de Soumission


        submission_df = pd.read_parquet(CHEMIN_SUBMISSION_TEMPLATE)

        submission_df['target'] = predictions
        submission_df = submission_df[['id', 'target']]

        submission_df.to_parquet(CHEMIN_SORTIE_PREDICTION, index=False)

        print("\n" + "="*50)
        print(" PRÉDICTION TERMINÉE AVEC SUCCÈS")
        print(f"Fichier de soumission sauvegardé à: {CHEMIN_SORTIE_PREDICTION}")
        print("="*50)

    except Exception as e:
        print(f"ERREUR: Échec du chargement ou de la prédiction du modèle.")
        print(f"Détails de l'erreur: {e}")
        # raise e # On ne lève pas l'exception pour afficher le message d'erreur personnalisé

Chargement des données test et application du Feature Engineering...
Features finales: 56
Dimensions X_test: (654068, 56)

Chargement du modèle LightGBM au format texte natif (.txt)...
ERREUR: Le fichier modèle n'existe pas à l'emplacement spécifié : /content/drive/MyDrive/lgbm_fold_4_auc_7191.txt
Veuillez vérifier le chemin d'accès dans votre Google Drive.
